# Germinal Antibody Design

![Germinal Antibody Design](https://proto-bio.github.io/proto-assets/images/tool/germinal/hero.png)

De novo epitope-targeted antibody design (VHH or scFv) using the Germinal pipeline ([Mille-Fragoso et al. 2025](https://www.biorxiv.org/content/10.1101/2025.09.19.677421)). This wrapper installs the upstream [`SantiagoMille/germinal`](https://github.com/SantiagoMille/germinal) repo at a pinned commit and runs one end-to-end campaign per call.

The pipeline is: **ColabDesign + AF2-Multimer hallucination → AbMPNN sequence redesign → Chai-1 / AF3 / Protenix structure validation → multi-stage filter cascade**.

> **Note: long-running tool.** Binder design campaigns take a long time on GPU, so the design cells below are left unexecuted and shown for reference.

In [1]:
from pathlib import Path

from proto_tools.tools.binder_design import (
    GerminalConfig,
    GerminalInput,
    run_germinal_design,
)
from proto_tools.utils.notebook_docs import display_api_reference


## Input API Reference

In [2]:
display_api_reference("germinal", "input")

**Input** — `GerminalInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>target_pdb</code> | <code>Structure</code> | required | Target structure |
| <code>target_chain</code> | <code>str</code> | <code>'A'</code> | Target chain ID(s); comma-separated for multi-chain targets |
| <code>binder_chain</code> | <code>str</code> | <code>'B'</code> | Chain ID assigned to the designed binder; must differ from target_chain. |
| <code>hotspots</code> | <code>list[str]</code> | <code>[]</code> | Target hotspot residues, e.g. ['A37','A39'] (chain letter + 1-indexed resnum) |
| <code>target_name</code> | <code>str &#124; None</code> | <code>None</code> | Short identifier; defaults to a hash of the PDB content |
| <code>hotspot_residue</code> | <code>str &#124; None</code> | <code>None</code> | Single-residue contact-restraint anchor for Chai-1 (e.g. 'W40'). Ignored by AF3 / Protenix. |

## Config API Reference

In [3]:
display_api_reference("germinal", "config")

**Config** — `GerminalConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device for the Germinal subprocess. Requires 'cuda' (CPU unsupported). |
| <code>timeout</code> | <code>int &#124; None</code> | <code>None</code> | Maximum execution time in seconds. None (default) waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>design_type</code> | <code>Literal['vhh', 'scfv']</code> | <code>'vhh'</code> | Run preset: 'vhh' (single-domain nanobody) or 'scfv' (scFv). |
| <code>max_trajectories</code> | <code>int</code> | <code>10000</code> | Hard cap on total trajectories before stopping (success or failure). |
| <code>max_hallucinated_trajectories</code> | <code>int</code> | <code>1000</code> | Cap on trajectories that complete the hallucination stage (before MPNN refinement). |
| <code>max_passing_designs</code> | <code>int</code> | <code>100</code> | Stop early once this many designs pass all final filters. |
| <code>structure_model</code> | <code>Literal['chai', 'af3', 'protenix']</code> | <code>'chai'</code> | Cofolding backend: 'chai' (auto-installed) or 'af3' / 'protenix' (user-provisioned). |
| <code>plddt_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Override final external_plddt (VHH preset: > 0.87, scFv preset: > 0.85). |
| <code>iptm_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Override final external_iptm (preset: > 0.74). |
| <code>ipae_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Override final external_pae (VHH preset: < 7.5, scFv preset: < 8). |
| <code>ptm_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Override final external_ptm (preset: > 0.84). |
| <code>pdockq2_threshold</code> | <code>float &#124; None</code> | <code>None</code> | Override final pdockq2 (preset: > 0.23). |
| <code>germinal_overrides</code> | <code>dict[str, Any]</code> | <code>{}</code> | Free-form Hydra overrides (key must be a valid dotpath). Applied as <key>=<value>. |
| <code>filter_overrides</code> | <code>dict[str, dict[str, dict[str, Any]]]</code> | <code>{}</code> | Override filter YAMLs: {'initial' \| 'final': {key: {'value': v, 'operator': op}}}. |
| <code>output_dir</code> | <code>str &#124; None</code> | <code>None</code> | Optional persistent output directory; if unset, a temp dir is used. |

## Output API Reference

In [4]:
display_api_reference("germinal", "output")

**Output** — `GerminalOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>designs</code> | <code>list[GerminalDesign]</code> | <code>[]</code> | All produced designs |
| <code>pipeline_stats</code> | <code>dict[str, int]</code> | <code>{}</code> | Per-stage counts from Germinal's failure_counts.csv |

**`GerminalDesign`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence_heavy</code> | <code>str</code> | required | Heavy chain or VHH amino acid sequence |
| <code>sequence_light</code> | <code>str &#124; None</code> | <code>None</code> | Light chain sequence (scFv only) |
| <code>structure</code> | <code>Structure</code> | required | Predicted binder + target complex |
| <code>metrics</code> | <code>GerminalDesignMetrics</code> | required | Per-design quality metrics from the Germinal pipeline. |
| <code>stage_passed</code> | <code>Literal['accepted', 'redesign_candidate', 'trajectory']</code> | required | Highest pipeline stage this design reached |
| <code>design_id</code> | <code>str</code> | required | Germinal's internal design identifier |
| <code>trajectory_index</code> | <code>int</code> | required | Trajectory seed; extracted from the '_s<seed>' suffix in design_id. |
| <code>mpnn_index</code> | <code>int</code> | required | AbMPNN sample index (1-based; 0 for trajectory-only designs without redesign). |

**Metrics** — `GerminalDesignMetrics`

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>plddt</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>ptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>i_ptm</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>i_pae</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>pae</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>loss</code> | <code>float</code> |  |  |  |
| <code>lm_ll</code> | <code>float</code> |  |  |  |
| <code>helix</code> | <code>float</code> |  |  |  |
| <code>beta_strand</code> | <code>float</code> |  |  |  |
| <code>clashes</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>sc_rmsd</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>binder_near_hotspot</code> | <code>bool</code> |  |  |  |
| <code>cdr3_hotspot_contacts</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>percent_interface_cdr</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>interface_shape_comp</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>interface_hbonds</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>surface_hydrophobicity</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>interface_hydrophobicity</code> | <code>float</code> | <code>&gt;= 0</code> |  |  |
| <code>pdockq2</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>external_plddt</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>external_iptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>external_ptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>external_pae</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>external_i_pae</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>external_i_plddt</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>external_plddt_binder</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>external_chain_ptm</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>external_binder_pae</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å</code> |  |
| <code>external_aggregate_score</code> | <code>float</code> |  |  |  |
| <code>ipsae</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>ipsae_pdockq2</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>lis_lis</code> | <code>float</code> |  |  |  |
| <code>lis_lia</code> | <code>float</code> |  |  |  |
| <code>binder_score</code> | <code>float</code> |  | <code>REU</code> |  |
| <code>interface_packstat</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>interface_dG</code> | <code>float</code> |  | <code>REU</code> |  |
| <code>interface_dSASA</code> | <code>float</code> | <code>&gt;= 0</code> | <code>Å²</code> |  |
| <code>interface_dG_SASA_ratio</code> | <code>float</code> |  | <code>REU/Å²</code> |  |
| <code>interface_fraction</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>interface_nres</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>interface_hbond_percentage</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>interface_delta_unsat_hbonds</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>interface_delta_unsat_hbonds_percentage</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>clashes_unrelaxed</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>hydrophobic_patches_binder</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>hydrophobic_patches_struct</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>sap_score</code> | <code>float</code> |  |  |  |
| <code>cdr_sap</code> | <code>float</code> |  |  |  |
| <code>cdr_hotspot_contacts</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>percent_interface_cdr3</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>alpha_interface</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>beta_interface</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>loops_interface</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>alpha_all</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>beta_all</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>loops_all</code> | <code>float</code> | <code>[0, 1]</code> |  |  |
| <code>n_framework_mutations</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |

In [ ]:
# Resolve the in-tree PD-L1 test target by walking up to the repo root.
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
pdb_path = _repo_root / "tests" / "dummy_data" / "pdl1.pdb"
print(f"Using target PDB: {pdb_path}")

inputs = GerminalInput(
    target_pdb=str(pdb_path),
    target_chain="A",
    binder_chain="B",
    hotspots=["A56", "A66", "A115"],
    target_name="pdl1",
)

config = GerminalConfig(
    design_type="vhh",
    max_trajectories=2,
    max_passing_designs=1,
    structure_model="chai",
    germinal_overrides={
        "logits_steps": 20,
        "softmax_steps": 4,
        "search_steps": 1,
        "num_seqs": 2,
    },
)

result = run_germinal_design(inputs, config)
print(f"success={result.success}")
print(f"num_designs={result.num_designs}, num_accepted={result.num_accepted}")
print(f"pipeline_stats={result.pipeline_stats}")

In [ ]:
for design in result.designs[:3]:
    print(f"--- {design.design_id} ({design.stage_passed}) ---")
    print(f"sequence_heavy: {design.sequence_heavy}")
    if design.sequence_light:
        print(f"sequence_light: {design.sequence_light}")
    if 'i_ptm' in design.metrics:
        print(f"  i_ptm={design.metrics.i_ptm:.3f}")
    if 'plddt' in design.metrics:
        print(f"  plddt={design.metrics.plddt:.3f}")

# Export every produced design's PDB (one file per design_id under <export_path>/<name.lower()>/)
if result.designs:
    result.export(name="pdl1_designs", export_path="./outputs", file_format="pdb")
    print(f"Exported {result.num_designs} designs ({result.num_accepted} accepted) to ./outputs/pdl1_designs/")

In [ ]:
# Advanced configuration: scFv mode + custom filter override + verbatim Hydra settings.
# This cell only constructs the config (no GPU / no subprocess) so the notebook
# can be executed end-to-end on hosts without GPU.
advanced_config = GerminalConfig(
    design_type="scfv",
    max_trajectories=10,
    max_passing_designs=2,
    structure_model="chai",
    iptm_threshold=0.80,  # Tighten the iptm filter beyond Germinal's preset (0.74)
    filter_overrides={
        "final": {
            # Require at least 4 H-bonds at the interface
            "interface_hbonds": {"value": 4, "operator": ">="},
        },
    },
    germinal_overrides={
        # Emphasise interface pTM in the hallucination loss
        "weights_iptm": 1.0,
        # Disable the helix bias for floppy / loopy CDR designs
        "weights_helix": 0.0,
    },
)

print("Advanced scFv config:")
print(f"  design_type        = {advanced_config.design_type}")
print(f"  structure_model    = {advanced_config.structure_model}")
print(f"  iptm_threshold     = {advanced_config.iptm_threshold}")
print(f"  filter_overrides   = {advanced_config.filter_overrides}")
print(f"  germinal_overrides = {advanced_config.germinal_overrides}")
# To actually run: result = run_germinal_design(inputs, advanced_config)